# GD - Homework 2

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 3. Implement GD, Polyak GD, Nesterov GD, and AdaGrad GD.

* GD: xk+1 = xk−γ∇f (xk )
* Polyak: xk+1 = xk−γ∇f (xk ) + µ(xk−xk−1)
* Nesterov: xk+1 = xk−γ∇f xk + µ(xk−xk−1) + µ(xk−xk−1)
* AdaGrad: xk+1 = xk−γDk ·∇f (xk )

In [ ]:
#NUMERIC helper for gradient (for harder problems where we don't have an analytical gradient)
def get_grad(f, x, eps=1e-5):
    g = np.zeros_like(x)
    for i in range(len(x)):
        x1, x2 = x.copy(), x.copy()
        x1[i] += eps; x2[i] -= eps
        g[i] = (f(x1) - f(x2)) / (2 * eps)
    return g

In [22]:
def gradient_descent(func, x0, gamma, n_iters, grad_f=None):
    if grad_f is None: grad_f = lambda x: get_grad(func, x)

    x = np.array(x0, dtype=float)
    history = [x.copy()]
    for _ in range(n_iters):
        grad = grad_f(x)
        x = x - gamma * grad
        history.append(x.copy())
    return np.array(history)

def polyak_gd(func, x0, gamma, mu, n_iters, grad_f=None):
    if grad_f is None: grad_f = lambda x: get_grad(func, x)

    x = np.array(x0, dtype=float)
    x_prev = x.copy()
    history = [x.copy()]
    for i in range(n_iters):
        grad = grad_f(x)
        if i == 0:
            #standard GD for first iter
            x_next = x - gamma * grad
        else:
            x_next = x - gamma * grad + mu * (x - x_prev)   
        x_prev = x.copy()
        x = x_next.copy()
        history.append(x.copy())  
    return np.array(history)

def nesterov_gd(func, x0, gamma, mu, n_iters, grad_f=None):
    if grad_f is None: grad_f = lambda x: get_grad(func, x)

    x = np.array(x0, dtype=float)
    x_prev = x.copy()
    history = [x.copy()]
    for i in range(n_iters):
        if i == 0:
            #standard GD
            x_next = x - gamma * grad_f(x)
        else:
            #momentum first
            x_lookahead = x + mu * (x - x_prev)
            x_next = x - gamma * grad_f(x_lookahead) + mu * (x - x_prev)           
        x_prev = x.copy()
        x = x_next.copy()
        history.append(x.copy())
    return np.array(history)

def adagrad_gd(func, x0, gamma, n_iters, epsilon=1e-8, grad_f=None):
    if grad_f is None: grad_f = lambda x: get_grad(func, x)

    x = np.array(x0, dtype=float)
    history = [x.copy()]
    # Cumulative sum of squared gradients for each coordinate
    sq_grad_sum = np.zeros_like(x)
    for _ in range(n_iters):
        grad = grad_f(x)
        sq_grad_sum += grad**2
        #Dk mtx
        D_k = 1.0 / (np.sqrt(sq_grad_sum) + epsilon)
        # component-wise update
        x = x - gamma * D_k * grad
        history.append(x.copy())
    return np.array(history)

**TEST:** on function $f(x,y)=x^2 + 5y^2$ with minimum at (0,0)... the same function as in the notes (fig 15,16)

**GRADIENT:** $\nabla f(x,y) = (2x, 10y)$. The Hessian has eigenvalues $\alpha = 2$ and $\beta = 10$.

**GD param:** optimal LR for a quadratic is $\gamma = \frac{2}{\alpha + \beta} = \frac{2}{12} \approx 0.166$.  

**Polyak params:** $\gamma = \frac{4}{(\sqrt{\alpha} + \sqrt{\beta})^2}$ and $\mu = \left(\frac{\sqrt{\beta} - \sqrt{\alpha}}{\sqrt{\beta} + \sqrt{\alpha}}\right)^2$. So we get $\gamma \approx 0.19$ and $\mu \approx 0.15$

**Nesterov params:** $\gamma = \frac{1}{\beta}$ and $\mu = \frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}$, where $\kappa = \frac{\beta}{\alpha}$... (theorem 5.3). So this gives us $\gamma = 0.1$ and $\mu \approx 0.382$

**AdaGrad param:** standard learning rate ($\gamma = 1.0$) because AdaGrad independently weights each coordinate by accumulating past gradients, meaning it adapts its own step size.

In [24]:
def f(v):
    x, y = v
    return x**2 + 5 * y**2

def grad_f(v):
    x, y = v
    return np.array([2*x, 10*y])

#initial params
x0 = [1.0, 1.0]
n_iterations = 10

#optimal params (based on alpha=2, beta=10)
gamma_gd = 1/6
gamma_polyak = 4 / (np.sqrt(2) + np.sqrt(10))**2
mu_polyak = ((np.sqrt(10) - np.sqrt(2)) / (np.sqrt(10) + np.sqrt(2)))**2
kappa = 10 / 2
gamma_nesterov = 1 / 10
mu_nesterov = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)

hist_gd = gradient_descent(f, x0, gamma=gamma_gd, n_iters=n_iterations, grad_f=grad_f)
hist_polyak = polyak_gd(f, x0, gamma=gamma_polyak, mu=mu_polyak, n_iters=n_iterations, grad_f=grad_f)
hist_nesterov = nesterov_gd(f, x0, gamma=gamma_nesterov, mu=mu_nesterov, n_iters=n_iterations, grad_f=grad_f)
hist_adagrad = adagrad_gd(f, x0, gamma=1.0, n_iters=n_iterations, grad_f=grad_f)

print("Final positions after 10 iterations (target: [0.0, 0.0]):\n")
print(f"Standard GD: {hist_gd[-1]} | f(x,y) = {f(hist_gd[-1]):.6f}")
print(f"Polyak GD:   {hist_polyak[-1]} | f(x,y) = {f(hist_polyak[-1]):.6f}")
print(f"Nesterov GD: {hist_nesterov[-1]} | f(x,y) = {f(hist_nesterov[-1]):.6f}")
print(f"AdaGrad GD:  {hist_adagrad[-1]} | f(x,y) = {f(hist_adagrad[-1]):.6f}")

Final positions after 10 iterations (target: [0.0, 0.0]):

Standard GD: [0.01734153 0.01734153] | f(x,y) = 0.001804
Polyak GD:   [0.00047467 0.00097968] | f(x,y) = 0.000005
Nesterov GD: [0.01457909 0.        ] | f(x,y) = 0.000213
AdaGrad GD:  [9.76562404e-84 1.00000069e-90] | f(x,y) = 0.000000


## 4. Find, describe, and implement Adam GD.

Adam (Adaptive Moment Estimation) combines the heavy-ball momentum from Polyak/Nesterov with the adaptive, coordinate-wise scaling of AdaGrad/RMSProp.  Instead of just adapting the learning rate based on the sum of past squared gradients (like AdaGrad, which eventually dampens the process too much ), Adam keeps an exponentially decaying average of past squared gradients. It also keeps an exponentially decaying average of past gradients (like momentum).

At each step k it calculates:
* First moment (mean): $m_k = \beta_1 m_{k-1} + (1 - \beta_1) \nabla f(x_{k-1})$
* Second moment (var): $v_k = \beta_2 v_{k-1} + (1 - \beta_2) (\nabla f(x_{k-1}))^2$

Bias correction step (to fix biasness toward 0):
* $\hat{m}_k = \frac{m_k}{1 - \beta_1^k}$
* $\hat{v}_k = \frac{v_k}{1 - \beta_2^k}$

Update rule:
$x_k = x_{k-1} - \gamma \frac{\hat{m}_k}{\sqrt{\hat{v}_k} + \epsilon}$

Params:
* $\gamma$ - global LR (npr 0.001)
* $\beta_1$ - exponential decay rate for first moment (npr 0.9)
* $\beta_2$ - exponential decay rate for second moment (npr 0.999)
* $\epsilon$ - prevent divison by 0

In [25]:
def adam_gd(func, x0, gamma=0.1, n_iters=100, beta1=0.9, beta2=0.999, epsilon=1e-8, grad_f=None):
    if grad_f is None: grad_f = lambda x: get_grad(func, x)
    
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    #first and second moments to zero
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    
    for k in range(1, n_iters + 1):
        grad = grad_f(x)
        #update biased first moment
        m = beta1 * m + (1 - beta1) * grad
        #ppdate biased second raw moment
        v = beta2 * v + (1 - beta2) * (grad**2)
        #compute bias-corrected first moment estimate
        m_hat = m / (1 - beta1**k)
        #compute bias-corrected second raw moment estimate
        v_hat = v / (1 - beta2**k)
        
        # update params
        x = x - gamma * m_hat / (np.sqrt(v_hat) + epsilon)
        history.append(x.copy())
        
    return np.array(history)

In [26]:
def f(v):
    x, y = v
    return x**2 + 5 * y**2

def grad_f(v):
    x, y = v
    return np.array([2*x, 10*y])

#initial params
x0 = [1.0, 1.0]
n_iterations = 100

#use LR of 0.5 for faster convergence on this simple quadratic
hist_adam = adam_gd(f, x0, gamma=0.5, n_iters=n_iterations, grad_f=grad_f)

print("Final position after 100 iterations (target: [0.0, 0.0]):\n")
print(f"Adam GD: {hist_adam[-1]} | f(x,y) = {f(hist_adam[-1]):.10f}")

Final position after 100 iterations (target: [0.0, 0.0]):

Adam GD: [-0.00434583 -0.00434583] | f(x,y) = 0.0001133174


## 5. Implement the Newton method and BFGS

NEWTON: $x_{k+1} = x_k - (\nabla^2 f(x_k))^{-1} \nabla f(x_k)$ 

BFGS: $B_{k+1} = B_k - \frac{\delta \gamma^T B_k + B_k \gamma \delta^T}{\delta^T \gamma} + \left(1 + \frac{\gamma^T B_k \gamma}{\delta^T \gamma}\right) \frac{\delta \delta^T}{\delta^T \gamma}$   

In [ ]:
#NUMERIC helper for hessian
def get_hess(f, x, eps=1e-5):
    n = len(x)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            x1, x2, x3, x4 = x.copy(), x.copy(), x.copy(), x.copy()
            x1[i] += eps; x1[j] += eps
            x2[i] += eps; x2[j] -= eps
            x3[i] -= eps; x3[j] += eps
            x4[i] -= eps; x4[j] -= eps
            H[i,j] = (f(x1) - f(x2) - f(x3) + f(x4)) / (4 * eps * eps)
    
    #check if PD, if not, apply modification
    eigvals = np.linalg.eigvalsh(H)
    if np.any(eigvals <= 0):
        H += (abs(np.min(eigvals)) + 1e-4) * np.eye(n)
    return H

In [27]:
#LINE SEARCH helper
def line_search(f, x, direction, grad):
    alpha = 1.0
    c = 1e-4
    rho = 0.5
    # Armijo condition backtracking
    while f(x + alpha * direction) > f(x) + c * alpha * np.dot(grad, direction):
        alpha *= rho
        if alpha < 1e-8: break # prevent infinite loops
    return alpha

In [28]:
def newton_method(func, x0, n_iters=10, grad_f=None, hessian_f=None):
    if grad_f is None: grad_f = lambda x: get_grad(func, x)
    if hessian_f is None: hessian_f = lambda x: get_hess(func, x)

    x = np.array(x0, dtype=float)
    history = [x.copy()]
    for _ in range(n_iters):
        grad = grad_f(x)
        if np.linalg.norm(grad) < 1e-6: break # converged
        H = hessian_f(x)
        #get direction of the step
        dir = -np.linalg.solve(H, grad)

        #apply line search
        alpha = line_search(func, x, dir, grad)
        x = x + alpha * dir
        history.append(x.copy())
        
    return np.array(history)


def bfgs_method(func, x0, n_iters=20, grad_f=None):
    if grad_f is None: grad_f = lambda x: get_grad(func, x)

    x = np.array(x0, dtype=float)
    n = len(x)
    #initialize B_0 as I (initial guess for inverse hessian)
    B = np.eye(n) 
    history = [x.copy()]
    
    for _ in range(n_iters):
        grad = grad_f(x)
        if np.linalg.norm(grad) < 1e-6: break
        #step direction
        step_dir = -np.dot(B, grad)

        #apply line search
        alpha = line_search(func, x, step_dir, grad)
        x_next = x + alpha * step_dir
        grad_next = grad_f(x_next)
        
        delta = x_next - x      #step diff
        gamma = grad_next - grad    #gradient diff
        #denominator
        denom = np.dot(delta, gamma)
        #avoid division by zero
        if denom > 1e-10:
            term1 = (np.outer(delta, gamma) @ B + B @ np.outer(gamma, delta)) / denom
            scalar_part = 1.0 + np.dot(gamma, B @ gamma) / denom
            term2 = scalar_part * (np.outer(delta, delta) / denom)
            #rank-2 update of the inverse hessian approx
            B = B - term1 + term2
            
        x = x_next.copy()
        history.append(x.copy())
        
    return np.array(history)

Again test on quadratic: $f(x,y) = x^2 + 5y^2$

In [30]:
def f(v):
    x, y = v
    return x**2 + 5 * y**2

def grad_f(v):
    x, y = v
    return np.array([2*x, 10*y])

def hessian_f(v):
    return np.array([[2, 0], 
                     [0, 10]])

x0 = [1.0, 1.0]

#newton
hist_newton = newton_method(f, x0, n_iters=5, grad_f=grad_f, hessian_f=hessian_f)

#BFGS
hist_bfgs = bfgs_method(f, x0, n_iters=20, grad_f=grad_f)

print("NEWTON:")
print(f"Iter 0: {hist_newton[0]} | f(x) = {f(hist_newton[0]):.4f}")
print(f"Iter 1: {hist_newton[1]} | f(x) = {f(hist_newton[1]):.4f}")
print(f"Iter 5: {hist_newton[-1]} | f(x) = {f(hist_newton[-1]):.4f}\n")

print("BFGS METHOD:")
print(f"Iter 0: {hist_bfgs[0]} | f(x) = {f(hist_bfgs[0]):.4f}")
print(f"Iter 1: {hist_bfgs[1]} | f(x) = {f(hist_bfgs[1]):.4f}")
print(f"Iter 5: {hist_bfgs[5]} | f(x) = {f(hist_bfgs[5]):.6f}")
print(f"Iter 20: {hist_bfgs[-1]} | f(x) = {f(hist_bfgs[-1]):.10f}")

NEWTON:
Iter 0: [1. 1.] | f(x) = 6.0000
Iter 1: [0. 0.] | f(x) = 0.0000
Iter 5: [0. 0.] | f(x) = 0.0000

BFGS METHOD:
Iter 0: [1. 1.] | f(x) = 6.0000
Iter 1: [ 0.75 -0.25] | f(x) = 0.8750
Iter 5: [-2.74263778e-06  4.57960411e-07] | f(x) = 0.000000
Iter 20: [ 1.66389840e-08 -4.75759014e-10] | f(x) = 0.0000000000


We can see that the Newton method converges instantly in one step - since our function is perfectly quadratic. BFGS method takes longer but also approaches the minimum very fast as B matrix is updated.

**Choice of $\alpha$**:

In notes it is said:

A line search is typically performed by verifying the Wolfe (or sometimes Armijo) conditions for a chosen α and then adjusting α if necessary. So first I tried solving this problem with $\alpha$ fixed at 0.2, which also made the BFGS converge just a bit slower. Then because of problem 6, I have rewritten this function to now use line search helper with Armijo condition to dynamically adjust $\alpha$.

## 6. Compare the methods of 3., 4, and 5. on:

In [19]:
import time

In [31]:
def func_a(v):
    x, y, z = v
    return (x - z)**2 + (2*y + z)**2 + (4*x - 2*y + z)**2 + x + y

def func_b(v):
    x, y, z = v
    return (x - 1)**2 + (y - 1)**2 + 100*(y - x**2)**2 + 100*(z - y**2)**2

def func_c(v):
    x, y = v
    return (1.5 - x + x*y)**2 + (2.25 - x + x*y**2)**2 + (2.625 - x + x*y**3)**2

In [32]:
tasks = [
    ("Function A", func_a, [0.0, 0.0, 0.0]),
    ("Function A", func_a, [1.0, 1.0, 0.0]),
    ("Function B", func_b, [1.2, 1.2, 1.2]),
    ("Function B", func_b, [-1.0, 1.2, 1.2]),
    ("Function C", func_c, [1.0, 1.0]),
    ("Function C", func_c, [4.5, 4.5])
]

steps_to_check = [2, 5, 10, 100]

for name, func, start in tasks:
    print(f"\n{'='*70}\n{name} | Start: {start}\n{'='*70}")
    
    methods = [
        ("GD", lambda it: gradient_descent(func, start, gamma=0.001, n_iters=it)),
        ("Polyak", lambda it: polyak_gd(func, start, gamma=0.001, mu=0.5, n_iters=it)),
        ("Nesterov", lambda it: nesterov_gd(func, start, gamma=0.001, mu=0.5, n_iters=it)),
        ("AdaGrad", lambda it: adagrad_gd(func, start, gamma=0.5, n_iters=it)),
        ("Adam", lambda it: adam_gd(func, start, gamma=0.1, n_iters=it)),
        ("Newton", lambda it: newton_method(func, start, n_iters=it)),
        ("BFGS", lambda it: bfgs_method(func, start, n_iters=it))
    ]
    
    print(f"{'Method':<12} | {'2 Steps':<10} | {'5 Steps':<10} | {'10 Steps':<10} | {'100 Steps':<10} | {'Time (100 iters)'}")
    print("-" * 80)
    
    for method_name, runner in methods:
        results = []
        time_taken = 0
        
        for step in steps_to_check:
            t0 = time.perf_counter()
            hist = runner(step)
            t1 = time.perf_counter()
            
            # Record time only for the 100-step run
            if step == 100:
                time_taken = t1 - t0
                
            final_val = func(hist[-1])
            results.append(final_val)
            
        print(f"{method_name:<12} | {results[0]:<10.4f} | {results[1]:<10.4f} | {results[2]:<10.4f} | {results[3]:<10.4f} | {time_taken:.4f}s")


Function A | Start: [0.0, 0.0, 0.0]
Method       | 2 Steps    | 5 Steps    | 10 Steps   | 100 Steps  | Time (100 iters)
--------------------------------------------------------------------------------
GD           | -0.0039    | -0.0096    | -0.0185    | -0.1114    | 0.0022s
Polyak       | -0.0049    | -0.0153    | -0.0317    | -0.1532    | 0.0016s
Nesterov     | -0.0049    | -0.0152    | -0.0315    | -0.1530    | 0.0015s
AdaGrad      | 5.6570     | -0.0900    | -0.1748    | -0.1979    | 0.0014s
Adam         | -0.1647    | -0.1613    | -0.1809    | -0.1979    | 0.0016s
Newton       | -0.1979    | -0.1979    | -0.1979    | -0.1979    | 0.0001s
BFGS         | -0.1339    | -0.1979    | -0.1979    | -0.1979    | 0.0003s

Function A | Start: [1.0, 1.0, 0.0]
Method       | 2 Steps    | 5 Steps    | 10 Steps   | 100 Steps  | Time (100 iters)
--------------------------------------------------------------------------------
GD           | 10.2427    | 9.2779     | 8.0179     | 1.9365     | 0.00